In [ ]:
MODEL_LABELS = {
    "deepseek_7b": "DeepSeek", "mistral_7b": "Mistral", "phi35_mini": "Phi-3.5",
    "gemma3_4b": "Gemma-3", "llama31_8b": "Llama-3.1", "qwen2_7b": "Qwen2",
    "olmo2_7b": "OLMo-2",
}

WARMTH = ["tolerant", "warm", "good-natured", "sincere"]
COMPETENCE = ["competent", "confident", "independent", "intelligent"]
ADJECTIVES = WARMTH + COMPETENCE
INPUT_GLOB = "output/simplified/experiment_results_*.csv"
MIN_N_CELLS = 5

In [ ]:
import glob, json, re
import pandas as pd
import numpy as np
import textstat
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 160)

def extract_json_object(text):
    if not isinstance(text, str): return None
    s = text.find("{")
    if s == -1: return None
    depth, ins, esc = 0, False, False
    for i in range(s, len(text)):
        c = text[i]
        if ins:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': ins = False
            continue
        if c == '"': ins = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0: return text[s:i+1]
    return None

def robust_parse(text):
    cand = extract_json_object(text)
    if cand is None: return None
    try: return json.loads(cand)
    except json.JSONDecodeError:
        rep = re.sub(r'(:\s*\d+),(\d+)(?=\s*[,}])', r'\1.\2', cand)
        try: return json.loads(rep)
        except json.JSONDecodeError: return None

def extract_adjectives(raw):
    d = robust_parse(raw)
    if d is None or "error_parsing" in d:
        return pd.Series([np.nan]*8, index=ADJECTIVES)
    return pd.Series([pd.to_numeric(d.get(a, np.nan), errors="coerce") for a in ADJECTIVES], index=ADJECTIVES)

def extract_behavior(raw):
    d = robust_parse(raw)
    if d is None or "error_parsing" in d:
        return pd.Series([np.nan, np.nan], index=["tgl", "success_prob"])
    text = str(d.get("transmitted_instructions", "") or "").strip()
    success = pd.to_numeric(d.get("success_probability_percent", np.nan), errors="coerce")
    if len(text) < 10:
        tgl = np.nan
    else:
        try:
            fk = textstat.flesch_kincaid_grade(text)
            fog = textstat.gunning_fog(text)
            cli = textstat.coleman_liau_index(text)
            tgl = float(np.mean([fk, fog, cli]))
        except Exception:
            tgl = np.nan
    return pd.Series([tgl, success], index=["tgl", "success_prob"])

def load_and_aggregate():
    files = glob.glob(INPUT_GLOB)
    if not files:
        raise FileNotFoundError(INPUT_GLOB)
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    scm_col = "scm_raw" if "scm_raw" in df.columns else "scm_json"
    beh_col = "behavior_raw" if "behavior_raw" in df.columns else "behavior_json"
    df[ADJECTIVES] = df[scm_col].apply(extract_adjectives)
    df[["tgl", "success_prob"]] = df[beh_col].apply(extract_behavior)

    agg_cols = ADJECTIVES + ["tgl", "success_prob"]
    cell = df.groupby(["model_id", "condition"])[agg_cols].mean().reset_index()
    n_cell = df.groupby(["model_id", "condition"]).size().rename("n").reset_index()
    cell = cell.merge(n_cell, on=["model_id", "condition"])

    thin = cell[cell["n"] < MIN_N_CELLS]
    if len(thin):
        print(f"WARNING: {len(thin)} cells with n < {MIN_N_CELLS}:")
        print(thin[["model_id", "condition", "n"]].to_string(index=False))
    return cell

def add_zscore(cell):
    cell = cell.copy()
    cell["z_success"] = cell.groupby("model_id")["success_prob"].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else np.nan
    )
    return cell

def run_correlations(cell, per_model=True):
    rows = []
    groups = cell.groupby("model_id") if per_model else [("ALL_POOLED", cell)]
    for model, g in groups:
        g = g.dropna(subset=["tgl", "z_success"])
        if len(g) < 4:
            continue
        for adj in ADJECTIVES:
            role = "competence (hypothesis-driven)" if adj in COMPETENCE else "warmth (contrast)"
            gv = g.dropna(subset=[adj])
            if len(gv) < 4:
                continue
            for dv_name, dv_col in [("tgl", "tgl"), ("z_success", "z_success")]:
                rho, p = stats.spearmanr(gv[adj], gv[dv_col])
                rows.append({"model_id": model, "adjective": adj, "role": role,
                            "dv": dv_name, "n_cells": len(gv), "rho": rho, "p_raw": p})
    out = pd.DataFrame(rows)
    if len(out):
        valid = out["p_raw"].notna()
        out["p_fdr"] = np.nan
        out["sig_fdr"] = False
        if valid.any():
            rej, pf, _, _ = multipletests(out.loc[valid, "p_raw"], alpha=0.05, method="fdr_bh")
            out.loc[valid, "p_fdr"] = pf
            out.loc[valid, "sig_fdr"] = rej
        out["relevant"] = out["sig_fdr"] & (out["rho"].abs() >= 0.3)
    return out



In [ ]:
cell = load_and_aggregate()
cell = add_zscore(cell)
cell.to_csv("rq2_cells.csv", index=False)

res = run_correlations(cell, per_model=True)
comp_rel = res[(res["relevant"]) & (res["role"].str.startswith("competence"))]
print(comp_rel[["model_id","adjective","dv","rho","p_fdr","n_cells"]].round(3).to_string(index=False)
      if len(comp_rel) else "  (none)")

res.to_csv("rq2_correlations.csv", index=False)

## MEDIATION ANALYSIS

In [ ]:
import glob, json, re, warnings
import pandas as pd
import numpy as np
import pingouin as pg
import textstat
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 170)

WARMTH = ["tolerant", "warm", "good-natured", "sincere"]
COMPETENCE = ["competent", "confident", "independent", "intelligent"]
ADJECTIVES = WARMTH + COMPETENCE
INPUT_GLOB = "output/simplified/experiment_results_*.csv"
N_BOOT = 2000
MIN_CELL_N = 3     

def extract_json_object(text):
    if not isinstance(text, str): return None
    s = text.find("{")
    if s == -1: return None
    depth, ins, esc = 0, False, False
    for i in range(s, len(text)):
        c = text[i]
        if ins:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': ins = False
            continue
        if c == '"': ins = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0: return text[s:i+1]
    return None

def robust_parse(text):
    cand = extract_json_object(text)
    if cand is None: return None
    try: return json.loads(cand)
    except json.JSONDecodeError:
        rep = re.sub(r'(:\s*\d+),(\d+)(?=\s*[,}])', r'\1.\2', cand)
        try: return json.loads(rep)
        except json.JSONDecodeError: return None

def extract_adjectives(raw):
    d = robust_parse(raw)
    if d is None or "error_parsing" in d:
        return pd.Series([np.nan]*8, index=ADJECTIVES)
    return pd.Series([pd.to_numeric(d.get(a, np.nan), errors="coerce") for a in ADJECTIVES], index=ADJECTIVES)

def extract_behavior(raw):
    d = robust_parse(raw)
    if d is None or "error_parsing" in d:
        return pd.Series([np.nan, np.nan], index=["tgl", "success_prob"])
    text = str(d.get("transmitted_instructions", "") or "").strip()
    success = pd.to_numeric(d.get("success_probability_percent", np.nan), errors="coerce")
    tgl = np.nan
    if len(text) >= 10:
        try:
            fk = textstat.flesch_kincaid_grade(text)
            fog = textstat.gunning_fog(text)
            cli = textstat.coleman_liau_index(text)
            tgl = float(np.mean([fk, fog, cli]))
        except Exception:
            pass
    return pd.Series([tgl, success], index=["tgl", "success_prob"])

def load_and_aggregate():
    files = glob.glob(INPUT_GLOB)
    if not files:
        raise FileNotFoundError(INPUT_GLOB)
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    scm_col = "scm_raw" if "scm_raw" in df.columns else "scm_json"
    beh_col = "behavior_raw" if "behavior_raw" in df.columns else "behavior_json"
    df[ADJECTIVES] = df[scm_col].apply(extract_adjectives)
    df[["tgl", "success_prob"]] = df[beh_col].apply(extract_behavior)

    df["competence_mean"] = df["competent"]
    df["warmth_mean"] = df["warm"]

    group_cols = ["model_id", "condition", "vignette_id"]
    agg_cols = ["competence_mean", "warmth_mean", "tgl", "success_prob"]
    cell = df.groupby(group_cols)[agg_cols].mean().reset_index()
    n_cell = df.groupby(group_cols).size().rename("n").reset_index()
    cell = cell.merge(n_cell, on=group_cols)

    thin = cell[cell["n"] < MIN_CELL_N]
    if len(thin):
        print(f"WARNING: {len(thin)} cells with n < {MIN_CELL_N}.")

    cell["z_success"] = cell.groupby("model_id")["success_prob"].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else np.nan
    )
    return cell

def _ols_coef(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta[1]

def _ols_coef_multi(X_cols, y):
    Xs = np.column_stack([np.ones(len(y))] + [np.asarray(c, dtype=float) for c in X_cols])
    beta, *_ = np.linalg.lstsq(Xs, np.asarray(y, dtype=float), rcond=None)
    return beta[1:]  

def stratified_bootstrap_mediation(d, x_col="is_condition", m_col="competence_mean",
                                   y_col="tgl", n_boot=N_BOOT, seed=42):
    rng = np.random.default_rng(seed)
    g0 = d[d[x_col] == 0].reset_index(drop=True)
    g1 = d[d[x_col] == 1].reset_index(drop=True)
    if len(g0) < 2 or len(g1) < 2:
        return None

    def point_estimate(dd):
        a = _ols_coef(dd[x_col], dd[m_col])                          
        b_coefs = _ols_coef_multi([dd[x_col], dd[m_col]], dd[y_col])  
        b = b_coefs[1]        
        c_direct = b_coefs[0] 
        return a, b, c_direct

    a_obs, b_obs, direct_obs = point_estimate(d)
    indirect_obs = a_obs * b_obs

    boot_indirect = []
    for _ in range(n_boot):
        r0 = g0.sample(n=len(g0), replace=True, random_state=rng.integers(1e9))
        r1 = g1.sample(n=len(g1), replace=True, random_state=rng.integers(1e9))
        rr = pd.concat([r0, r1], ignore_index=True)
        if rr[m_col].nunique() < 2 or rr[y_col].nunique() < 2:
            continue
        try:
            a_b, b_b, _ = point_estimate(rr)
            boot_indirect.append(a_b * b_b)
        except Exception:
            continue

    if len(boot_indirect) < n_boot * 0.5:
        return {"degenerate": True, "n_valid_boot": len(boot_indirect)}

    boot_indirect = np.array(boot_indirect)
    ci_low, ci_high = np.percentile(boot_indirect, [2.5, 97.5])
    return {
        "degenerate": False,
        "path_a_coef": a_obs, "path_b_coef": b_obs,
        "indirect_coef": indirect_obs,
        "indirect_CI2.5": ci_low, "indirect_CI97.5": ci_high,
        "indirect_sig": not (ci_low <= 0 <= ci_high),
        "direct_coef": direct_obs,
        "n_valid_boot": len(boot_indirect),
    }

def run_mediation(cell):
    rows = []
    models = cell["model_id"].unique()
    conditions = [c for c in cell["condition"].unique() if c != "control"]

    for model in models:
        dm = cell[cell["model_id"] == model]
        ctrl = dm[dm["condition"] == "control"]
        if len(ctrl) < 2:
            continue
        for cond in conditions:
            grp = dm[dm["condition"] == cond]
            if len(grp) < 2:
                continue
            sub = pd.concat([
                ctrl.assign(is_condition=0),
                grp.assign(is_condition=1),
            ], ignore_index=True)

            for dv in ["tgl", "z_success"]:
                d = sub.dropna(subset=["competence_mean", dv, "is_condition"])
                if len(d) < 6 or d["competence_mean"].nunique() < 2 or d[dv].nunique() < 2:
                    rows.append({"model_id": model, "condition": cond, "dv": dv,
                                "n": len(d), "degenerate": True})
                    continue

                res = stratified_bootstrap_mediation(d, "is_condition", "competence_mean", dv)
                if res is None or res.get("degenerate"):
                    rows.append({"model_id": model, "condition": cond, "dv": dv,
                                "n": len(d), "degenerate": True,
                                "n_valid_boot": (res or {}).get("n_valid_boot", 0)})
                    continue

                rows.append({"model_id": model, "condition": cond, "dv": dv, "n": len(d),
                            "degenerate": False, **{k: v for k, v in res.items()
                                                    if k not in ("degenerate",)}})
    out = pd.DataFrame(rows)
    return out


In [ ]:
cell = load_and_aggregate()
cell.to_csv("rq2_mediation_cells.csv", index=False)


res = run_mediation(cell)
if len(res):
    n_degenerate = res["degenerate"].sum()
    valid = res[~res["degenerate"]].copy()
    print(f"\nTest: {len(res)} | failed: {n_degenerate} | valid: {len(valid)}")

    if len(valid):
        print(f"\nSignificant effect: "
              f"{valid['indirect_sig'].sum()} / {len(valid)}")
        print(valid[["model_id", "condition", "dv", "n", "path_a_coef", "path_b_coef",
                     "indirect_coef", "indirect_CI2.5", "indirect_CI97.5", "indirect_sig"]]
              .round(3).to_string(index=False))
    if n_degenerate:
        print(f"\n{n_degenerate} No valid results")

    res.to_csv("rq2_mediation_results.csv", index=False)